In [1]:
import pandas as pd
import re

In [12]:
seasons_wages_raw = pd.read_csv("../uncleaned_data_csv/seasons_wages.csv")

In [13]:
seasons_wages_raw.head()

,Rk,Squad,Competition,# of Players,Weekly Wages,Annual Wages,% Estimated,Season
0,1,Barcelona,es La Liga,33,"€ 4,682,115 (£ 3,925,993, $4,771,599)","€ 243,470,000 (£ 204,151,634, $248,123,125)",100%,2017-2018
1,2,Real Madrid,es La Liga,33,"€ 3,949,904 (£ 3,312,027, $4,025,393)","€ 205,395,000 (£ 172,225,426, $209,320,446)",100%,2017-2018
2,3,Paris S-G,fr Ligue 1,35,"€ 3,895,462 (£ 3,266,377, $3,969,910)","€ 202,564,000 (£ 169,851,608, $206,435,344)",100%,2017-2018
3,4,Manchester Utd,eng Premier League,35,"€ 3,810,955 (£ 3,195,596, $3,883,356)","€ 198,169,670 (£ 166,171,000, $201,934,520)",100%,2017-2018
4,5,Arsenal,eng Premier League,45,"€ 3,629,433 (£ 3,043,385, $3,698,385)","€ 188,730,521 (£ 158,256,000, $192,316,043)",100%,2017-2018


In [6]:
# Rename Competition to League
if "Competition" in seasons_wages_raw.columns:
    seasons_wages_raw = seasons_wages_raw.rename(columns={"Competition": "League"})
    seasons_wages_raw["League"] = seasons_wages_raw["League"].astype(str).str.extract(r"([A-Z][\w\s]+)")

# Drop rank column
if "Rk" in seasons_wages_raw.columns:
    seasons_wages_raw = seasons_wages_raw.drop(columns=["Rk"])

seasons_wages_raw.head()


,Squad,League,# of Players,Weekly Wages,Annual Wages,% Estimated,Season
0,Barcelona,La Liga,33,"€ 4,682,115 (£ 3,925,993, $4,771,599)","€ 243,470,000 (£ 204,151,634, $248,123,125)",100%,2017-2018
1,Real Madrid,La Liga,33,"€ 3,949,904 (£ 3,312,027, $4,025,393)","€ 205,395,000 (£ 172,225,426, $209,320,446)",100%,2017-2018
2,Paris S-G,Ligue 1,35,"€ 3,895,462 (£ 3,266,377, $3,969,910)","€ 202,564,000 (£ 169,851,608, $206,435,344)",100%,2017-2018
3,Manchester Utd,Premier League,35,"€ 3,810,955 (£ 3,195,596, $3,883,356)","€ 198,169,670 (£ 166,171,000, $201,934,520)",100%,2017-2018
4,Arsenal,Premier League,45,"€ 3,629,433 (£ 3,043,385, $3,698,385)","€ 188,730,521 (£ 158,256,000, $192,316,043)",100%,2017-2018


In [9]:
def extract_currency_values(s):
    if pd.isna(s):
        return {"EUR": None, "GBP": None, "USD": None}
    
    s = str(s)

    # Handle common encoding artifacts
    s = s.replace("â‚¬", "€").replace("Â£", "£").replace("Ł", "£")

    eur = gbp = usd = None

    eur_match = re.search(r"€\s*([\d,]+)", s)
    gbp_match = re.search(r"£\s*([\d,]+)", s)
    usd_match = re.search(r"\$\s*([\d,]+)", s)

    if eur_match:
        eur = float(eur_match.group(1).replace(",", ""))
    if gbp_match:
        gbp = float(gbp_match.group(1).replace(",", ""))
    if usd_match:
        usd = float(usd_match.group(1).replace(",", ""))

    return {"EUR": eur, "GBP": gbp, "USD": usd}


currency_data = seasons_wages_raw["Annual Wages"].apply(extract_currency_values)
currency_data.head()

0    {'EUR': 243470000.0, 'GBP': 204151634.0, 'USD'...
1    {'EUR': 205395000.0, 'GBP': 172225426.0, 'USD'...
2    {'EUR': 202564000.0, 'GBP': 169851608.0, 'USD'...
3    {'EUR': 198169670.0, 'GBP': 166171000.0, 'USD'...
4    {'EUR': 188730521.0, 'GBP': 158256000.0, 'USD'...
Name: Annual Wages, dtype: object

In [10]:
def add_wage_columns(df, wage_col, prefix):
    parsed = df[wage_col].apply(extract_currency_values)
    df[f"{prefix}_wages_eur"] = parsed.apply(lambda x: x["EUR"])
    df[f"{prefix}_wages_gbp"] = parsed.apply(lambda x: x["GBP"])
    df[f"{prefix}_wages_usd"] = parsed.apply(lambda x: x["USD"])
    return df

seasons_wages_raw = add_wage_columns(seasons_wages_raw, "Weekly Wages", "weekly")
seasons_wages_raw = add_wage_columns(seasons_wages_raw, "Annual Wages", "annual")

seasons_wages_raw.head()

,Squad,League,# of Players,Weekly Wages,Annual Wages,% Estimated,Season,weekly_wages_eur,weekly_wages_gbp,weekly_wages_usd,annual_wages_eur,annual_wages_gbp,annual_wages_usd
0,Barcelona,La Liga,33,"€ 4,682,115 (£ 3,925,993, $4,771,599)","€ 243,470,000 (£ 204,151,634, $248,123,125)",100%,2017-2018,4682115.0,3925993.0,4771599.0,243470000.0,204151634.0,248123125.0
1,Real Madrid,La Liga,33,"€ 3,949,904 (£ 3,312,027, $4,025,393)","€ 205,395,000 (£ 172,225,426, $209,320,446)",100%,2017-2018,3949904.0,3312027.0,4025393.0,205395000.0,172225426.0,209320446.0
2,Paris S-G,Ligue 1,35,"€ 3,895,462 (£ 3,266,377, $3,969,910)","€ 202,564,000 (£ 169,851,608, $206,435,344)",100%,2017-2018,3895462.0,3266377.0,3969910.0,202564000.0,169851608.0,206435344.0
3,Manchester Utd,Premier League,35,"€ 3,810,955 (£ 3,195,596, $3,883,356)","€ 198,169,670 (£ 166,171,000, $201,934,520)",100%,2017-2018,3810955.0,3195596.0,3883356.0,198169670.0,166171000.0,201934520.0
4,Arsenal,Premier League,45,"€ 3,629,433 (£ 3,043,385, $3,698,385)","€ 188,730,521 (£ 158,256,000, $192,316,043)",100%,2017-2018,3629433.0,3043385.0,3698385.0,188730521.0,158256000.0,192316043.0


In [11]:
if "% Estimated" in seasons_wages_raw.columns:
    seasons_wages_raw["pct_estimated"] = (
        seasons_wages_raw["% Estimated"].astype(str).str.replace("%", "", regex=False).str.strip()
    )
    seasons_wages_raw["pct_estimated"] = pd.to_numeric(seasons_wages_raw["pct_estimated"], errors="coerce").astype("float")

# Numeric / categorical
if "# of Players" in seasons_wages_raw.columns:
    seasons_wages_raw["# of Players"] = pd.to_numeric(seasons_wages_raw["# of Players"], errors="coerce").astype("Int64")

for c in ["League", "Squad", "Season"]:
    if c in seasons_wages_raw.columns:
        seasons_wages_raw[c] = seasons_wages_raw[c].astype("category")

seasons_wages_raw.dtypes

Squad               category
League              category
# of Players           Int64
Weekly Wages             str
Annual Wages             str
% Estimated              str
Season              category
weekly_wages_eur     float64
weekly_wages_gbp     float64
weekly_wages_usd     float64
annual_wages_eur     float64
annual_wages_gbp     float64
annual_wages_usd     float64
pct_estimated        float64
dtype: object

In [14]:
def clean_seasons_wages(df: pd.DataFrame) -> pd.DataFrame:
    # Work on a copy
    df = df.copy()

    # 1. Rename Competition -> League and clean league text
    if "Competition" in df.columns:
        df = df.rename(columns={"Competition": "League"})
    if "League" in df.columns:
        df["League"] = df["League"].astype(str).str.extract(r"([A-Z][\w\s]+)")

    # 2. Drop rank
    if "Rk" in df.columns:
        df = df.drop(columns=["Rk"])

    # 3. Parse wages into EUR/GBP/USD columns
    def _extract_currency_values(s):
        if pd.isna(s):
            return {"EUR": None, "GBP": None, "USD": None}

        s = str(s).replace("â‚¬", "€").replace("Â£", "£").replace("Ł", "£")

        eur_match = re.search(r"€\s*([\d,]+)", s)
        gbp_match = re.search(r"£\s*([\d,]+)", s)
        usd_match = re.search(r"\$\s*([\d,]+)", s)

        return {
            "EUR": float(eur_match.group(1).replace(",", "")) if eur_match else None,
            "GBP": float(gbp_match.group(1).replace(",", "")) if gbp_match else None,
            "USD": float(usd_match.group(1).replace(",", "")) if usd_match else None,
        }

    if "Weekly Wages" in df.columns:
        weekly = df["Weekly Wages"].apply(_extract_currency_values)
        df["weekly_wages_eur"] = weekly.apply(lambda x: x["EUR"])
        df["weekly_wages_gbp"] = weekly.apply(lambda x: x["GBP"])
        df["weekly_wages_usd"] = weekly.apply(lambda x: x["USD"])

    if "Annual Wages" in df.columns:
        annual = df["Annual Wages"].apply(_extract_currency_values)
        df["annual_wages_eur"] = annual.apply(lambda x: x["EUR"])
        df["annual_wages_gbp"] = annual.apply(lambda x: x["GBP"])
        df["annual_wages_usd"] = annual.apply(lambda x: x["USD"])

    # 4. Type conversions
    if "# of Players" in df.columns:
        df["# of Players"] = pd.to_numeric(df["# of Players"], errors="coerce").astype("Int64")

    if "% Estimated" in df.columns:
        df["pct_estimated"] = (
            df["% Estimated"].astype(str).str.replace("%", "", regex=False).str.strip()
        )
        df["pct_estimated"] = pd.to_numeric(df["pct_estimated"], errors="coerce").astype("float")

    for col in [
        "weekly_wages_eur", "weekly_wages_gbp", "weekly_wages_usd",
        "annual_wages_eur", "annual_wages_gbp", "annual_wages_usd"
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float")

    for col in ["League", "Squad", "Season"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    return df

In [15]:
wages_cleaned = clean_seasons_wages(seasons_wages_raw)
wages_cleaned.head()

,Squad,League,# of Players,Weekly Wages,Annual Wages,% Estimated,Season,weekly_wages_eur,weekly_wages_gbp,weekly_wages_usd,annual_wages_eur,annual_wages_gbp,annual_wages_usd,pct_estimated
0,Barcelona,La Liga,33,"€ 4,682,115 (£ 3,925,993, $4,771,599)","€ 243,470,000 (£ 204,151,634, $248,123,125)",100%,2017-2018,4682115.0,3925993.0,4771599.0,243470000.0,204151634.0,248123125.0,100.0
1,Real Madrid,La Liga,33,"€ 3,949,904 (£ 3,312,027, $4,025,393)","€ 205,395,000 (£ 172,225,426, $209,320,446)",100%,2017-2018,3949904.0,3312027.0,4025393.0,205395000.0,172225426.0,209320446.0,100.0
2,Paris S-G,Ligue 1,35,"€ 3,895,462 (£ 3,266,377, $3,969,910)","€ 202,564,000 (£ 169,851,608, $206,435,344)",100%,2017-2018,3895462.0,3266377.0,3969910.0,202564000.0,169851608.0,206435344.0,100.0
3,Manchester Utd,Premier League,35,"€ 3,810,955 (£ 3,195,596, $3,883,356)","€ 198,169,670 (£ 166,171,000, $201,934,520)",100%,2017-2018,3810955.0,3195596.0,3883356.0,198169670.0,166171000.0,201934520.0,100.0
4,Arsenal,Premier League,45,"€ 3,629,433 (£ 3,043,385, $3,698,385)","€ 188,730,521 (£ 158,256,000, $192,316,043)",100%,2017-2018,3629433.0,3043385.0,3698385.0,188730521.0,158256000.0,192316043.0,100.0
